# YZM304 Derin Öğrenme – II. Proje Ödevi
## CNN ile Özellik Çıkarma ve Sınıflandırma
**Veri Seti:** MNIST (Model 1-3) | CIFAR-10 (Model 4-5)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from collections import OrderedDict
import warnings
warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("Device :", device)


## 0. Hiperparametreler

In [ ]:
BATCH_SIZE = 64     # GPU/CPU belleğe uygun; iyi genelleme sağlar
LR_CNN    = 1e-3    # Adam optimizer için standart başlangıç öğrenme oranı
EPOCHS_12 = 10      # MNIST'te LeNet hızlı yakınsar
EPOCHS_3  = 5       # Büyük VGG modeli, küçük alt-küme üzerinde
EPOCHS_5  = 10      # CIFAR-10 CNN
SEED      = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
print("Sabitler ayarlandı.")


## 1. Veri Seti – MNIST (Model 1, 2, 3)

In [ ]:
transform_mnist = transforms.Compose([
    transforms.Pad(2),             # 28x28 -> 32x32 (LeNet-5 için)
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_mnist = torchvision.datasets.MNIST(root="./data", train=True,  download=True, transform=transform_mnist)
test_mnist  = torchvision.datasets.MNIST(root="./data", train=False, download=True, transform=transform_mnist)

train_loader_mnist = DataLoader(train_mnist, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
test_loader_mnist  = DataLoader(test_mnist,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"MNIST Train: {len(train_mnist):,}  |  Test: {len(test_mnist):,}")
print(f"Ornek sekli: {train_mnist[0][0].shape}")

fig, axes = plt.subplots(1, 8, figsize=(16, 2))
for i, ax in enumerate(axes):
    img, lbl = train_mnist[i]
    ax.imshow(img.squeeze(), cmap="gray")
    ax.set_title(str(lbl)); ax.axis("off")
plt.suptitle("MNIST Ornekleri", y=1.05)
plt.tight_layout(); plt.savefig("mnist_samples.png", bbox_inches="tight"); plt.show()


## 2. Model 1 – LeNet-5 Benzeri CNN

In [ ]:
class LeNet5(nn.Module):
    """
    LeNet-5 (LeCun et al., 1998) benzeri CNN.
    Giris: 1x32x32  |  Cikis: 10 sinif
    Katmanlar: Conv -> Tanh -> AvgPool -> Conv -> Tanh -> AvgPool -> Conv -> Flatten -> FC -> FC
    """
    def __init__(self, num_classes=10):
        super(LeNet5, self).__init__()
        self.conv1 = nn.Conv2d(1,  6,   kernel_size=5)
        self.pool1 = nn.AvgPool2d(kernel_size=2, stride=2)
        self.conv2 = nn.Conv2d(6,  16,  kernel_size=5)
        self.pool2 = nn.AvgPool2d(kernel_size=2, stride=2)
        self.conv3 = nn.Conv2d(16, 120, kernel_size=5)
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(120, 84)
        self.fc2 = nn.Linear(84, num_classes)

    def forward(self, x):
        x = torch.tanh(self.conv1(x)); x = self.pool1(x)
        x = torch.tanh(self.conv2(x)); x = self.pool2(x)
        x = torch.tanh(self.conv3(x))
        x = self.flatten(x)
        x = torch.tanh(self.fc1(x))
        return self.fc2(x)

model1 = LeNet5().to(device)
print(model1)
print(f"Egit. param: {sum(p.numel() for p in model1.parameters() if p.requires_grad):,}")


In [ ]:
def train_model(model, train_loader, test_loader, epochs, lr, label):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    history = {"train_loss": [], "train_acc": [], "test_acc": []}
    for epoch in range(1, epochs + 1):
        model.train()
        run_loss, correct, total = 0.0, 0, 0
        for imgs, lbls in train_loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            optimizer.zero_grad()
            out  = model(imgs)
            loss = criterion(out, lbls)
            loss.backward(); optimizer.step()
            run_loss += loss.item() * imgs.size(0)
            correct  += out.argmax(1).eq(lbls).sum().item()
            total    += lbls.size(0)
        trn_loss = run_loss / total
        trn_acc  = 100. * correct / total
        model.eval(); cor2, tot2 = 0, 0
        with torch.no_grad():
            for imgs, lbls in test_loader:
                imgs, lbls = imgs.to(device), lbls.to(device)
                cor2 += model(imgs).argmax(1).eq(lbls).sum().item()
                tot2 += lbls.size(0)
        tst_acc = 100. * cor2 / tot2
        history["train_loss"].append(trn_loss)
        history["train_acc"].append(trn_acc)
        history["test_acc"].append(tst_acc)
        print(f"[{label}] Epoch {epoch:02d}/{epochs} | Loss:{trn_loss:.4f} | Train:{trn_acc:.2f}% | Test:{tst_acc:.2f}%")
    return history

print("Egitim fonksiyonu hazir.")


In [ ]:
hist1 = train_model(model1, train_loader_mnist, test_loader_mnist, EPOCHS_12, LR_CNN, 'LeNet5')

## 3. Model 2 – LeNet-5 + BatchNorm + Dropout

In [ ]:
class LeNet5BN(nn.Module):
    """
    Model 1 ile ayni hiper-parametreler.
    Eklemeler: BatchNorm2d (normalizasyon, daha hizli yakinsama) + Dropout(0.5) (regularizasyon).
    """
    def __init__(self, num_classes=10):
        super(LeNet5BN, self).__init__()
        self.conv1 = nn.Conv2d(1,  6,   kernel_size=5)
        self.bn1   = nn.BatchNorm2d(6)
        self.pool1 = nn.AvgPool2d(2, 2)
        self.conv2 = nn.Conv2d(6,  16,  kernel_size=5)
        self.bn2   = nn.BatchNorm2d(16)
        self.pool2 = nn.AvgPool2d(2, 2)
        self.conv3 = nn.Conv2d(16, 120, kernel_size=5)
        self.bn3   = nn.BatchNorm2d(120)
        self.flatten = nn.Flatten()
        self.drop  = nn.Dropout(0.5)
        self.fc1   = nn.Linear(120, 84)
        self.fc2   = nn.Linear(84, num_classes)

    def forward(self, x):
        x = torch.tanh(self.bn1(self.conv1(x))); x = self.pool1(x)
        x = torch.tanh(self.bn2(self.conv2(x))); x = self.pool2(x)
        x = torch.tanh(self.bn3(self.conv3(x)))
        x = self.flatten(x); x = self.drop(x)
        x = torch.tanh(self.fc1(x)); x = self.drop(x)
        return self.fc2(x)

model2 = LeNet5BN().to(device)
print(model2)
print(f"Egit. param: {sum(p.numel() for p in model2.parameters() if p.requires_grad):,}")


In [ ]:
hist2 = train_model(model2, train_loader_mnist, test_loader_mnist, EPOCHS_12, LR_CNN, 'LeNet5BN')

## 4. Model 3 – VGG-11 (torchvision, pretrained=False)

In [ ]:
transform_vgg = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,)*3, (0.5,)*3)
])
tr_vgg = torchvision.datasets.MNIST("./data", train=True,  download=True, transform=transform_vgg)
te_vgg = torchvision.datasets.MNIST("./data", train=False, download=True, transform=transform_vgg)
train_loader_vgg = DataLoader(Subset(tr_vgg, range(10000)), batch_size=BATCH_SIZE, shuffle=True)
test_loader_vgg  = DataLoader(Subset(te_vgg, range(2000)),  batch_size=BATCH_SIZE, shuffle=False)

model3 = models.vgg11(weights=None)
model3.classifier[6] = nn.Linear(4096, 10)
model3 = model3.to(device)
print("VGG-11 yuklendi (pretrained=False)")
print(f"Egit. param: {sum(p.numel() for p in model3.parameters() if p.requires_grad):,}")


In [ ]:
hist3 = train_model(model3, train_loader_vgg, test_loader_vgg, EPOCHS_3, LR_CNN*0.1, 'VGG11')

## 5. Model 1-2-3 Karşılaştırma Grafikleri

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(hist1["train_loss"], "o-", label="LeNet-5")
axes[0].plot(hist2["train_loss"], "s-", label="LeNet5+BN+DO")
axes[0].set(title="Egitim Loss", xlabel="Epoch", ylabel="CrossEntropy")
axes[0].legend(); axes[0].grid(True)

axes[1].plot(hist1["test_acc"], "o-", label="LeNet-5")
axes[1].plot(hist2["test_acc"], "s-", label="LeNet5+BN+DO")
axes[1].plot(range(EPOCHS_3), hist3["test_acc"], "^-", label="VGG-11")
axes[1].set(title="Test Dogrulugu (%)", xlabel="Epoch", ylabel="Accuracy")
axes[1].legend(); axes[1].grid(True)

plt.suptitle("Model 1-2-3 Karsilastirmasi (MNIST)", fontsize=13)
plt.tight_layout()
plt.savefig("model123_comparison.png", bbox_inches="tight"); plt.show()

print(f"Model1 (LeNet-5)   test acc: {hist1['test_acc'][-1]:.2f}%")
print(f"Model2 (BN+Drop)   test acc: {hist2['test_acc'][-1]:.2f}%")
print(f"Model3 (VGG-11)    test acc: {hist3['test_acc'][-1]:.2f}%")


## 6. Karmaşıklık Matrisleri (Model 1-2)

In [ ]:
def get_preds(model, loader):
    model.eval(); all_p, all_t = [], []
    with torch.no_grad():
        for imgs, lbls in loader:
            imgs = imgs.to(device)
            all_p.extend(model(imgs).argmax(1).cpu().numpy())
            all_t.extend(lbls.numpy())
    return np.array(all_t), np.array(all_p)

y_true1, y_pred1 = get_preds(model1, test_loader_mnist)
y_true2, y_pred2 = get_preds(model2, test_loader_mnist)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, yt, yp, title in zip(axes,
        [y_true1, y_true2], [y_pred1, y_pred2],
        ["LeNet-5", "LeNet5+BN+DO"]):
    cm = confusion_matrix(yt, yp)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax)
    ax.set(title=f"Karmasiklik Matrisi - {title}",
           xlabel="Tahmin", ylabel="Gercek")
plt.tight_layout()
plt.savefig("confusion_matrices.png", bbox_inches="tight"); plt.show()

print("\nModel1 Siniflandirma Raporu:")
print(classification_report(y_true1, y_pred1, digits=4))
print("\nModel2 Siniflandirma Raporu:")
print(classification_report(y_true2, y_pred2, digits=4))


## 7. Model 4 – Hibrit: CNN Özellik Çıkarıcı + SVM (CIFAR-10)

In [ ]:
# CIFAR-10 veri seti
transform_cifar = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2023, 0.1994, 0.2010))
])
tr_cifar = torchvision.datasets.CIFAR10("./data", train=True,  download=True, transform=transform_cifar)
te_cifar = torchvision.datasets.CIFAR10("./data", train=False, download=True, transform=transform_cifar)

# Hiz icin alt-kume
sub_tr = Subset(tr_cifar, range(8000))
sub_te = Subset(te_cifar, range(2000))
loader_tr_c = DataLoader(sub_tr, batch_size=BATCH_SIZE, shuffle=True)
loader_te_c = DataLoader(sub_te, batch_size=BATCH_SIZE, shuffle=False)

CIFAR_CLASSES = ["airplane","automobile","bird","cat","deer",
                 "dog","frog","horse","ship","truck"]
print(f"CIFAR-10 alt-kume -> Train: {len(sub_tr):,} | Test: {len(sub_te):,}")
print(f"Siniflar: {CIFAR_CLASSES}")


In [ ]:
class CIFAR_CNN(nn.Module):
    """
    CIFAR-10 icin CNN oznitelik cikarici.
    Son FC katmani olmadan ozellik vektoru cikartir.
    """
    def __init__(self, num_classes=10):
        super(CIFAR_CNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32,  kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2,2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2,2),
            nn.Conv2d(64,128, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2,2),
        )
        self.flatten = nn.Flatten()
        self.classifier = nn.Sequential(
            nn.Linear(128*4*4, 256), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.flatten(x)
        return self.classifier(x)

    def extract_features(self, x):
        x = self.features(x)
        return self.flatten(x)

model5_cnn = CIFAR_CNN().to(device)
print(model5_cnn)
print(f"Egit. param: {sum(p.numel() for p in model5_cnn.parameters() if p.requires_grad):,}")


### 7a. Model 5 – Tam CNN (CIFAR-10 baseline)

In [ ]:
hist5 = train_model(model5_cnn, loader_tr_c, loader_te_c, EPOCHS_5, LR_CNN, 'CIFAR_CNN')

### 7b. Özellik Çıkarma ve .npy Dosyaları

In [ ]:
def extract_features(model, loader):
    model.eval(); feats, labels = [], []
    with torch.no_grad():
        for imgs, lbls in loader:
            imgs = imgs.to(device)
            f = model.extract_features(imgs).cpu().numpy()
            feats.append(f); labels.append(lbls.numpy())
    return np.vstack(feats), np.concatenate(labels)

X_train, y_train = extract_features(model5_cnn, loader_tr_c)
X_test,  y_test  = extract_features(model5_cnn, loader_te_c)

np.save("X_train_features.npy", X_train)
np.save("y_train_labels.npy",   y_train)
np.save("X_test_features.npy",  X_test)
np.save("y_test_labels.npy",    y_test)

print(f"X_train sekli : {X_train.shape}  (ornek, ozellik)")
print(f"y_train sekli : {y_train.shape}")
print(f"X_test  sekli : {X_test.shape}")
print(f"y_test  sekli : {y_test.shape}")
print(".npy dosyalari kaydedildi.")


### 7c. SVM Sınıflandırıcı (Model 4)

In [ ]:
# SVM: RBF kernel, C=10
svm = SVC(kernel="rbf", C=10, gamma="scale", random_state=SEED)
svm.fit(X_train, y_train)
y_pred_svm = svm.predict(X_test)

acc_svm = accuracy_score(y_test, y_pred_svm) * 100
print(f"SVM Test Dogrulugu: {acc_svm:.2f}%")
print(classification_report(y_test, y_pred_svm, target_names=CIFAR_CLASSES, digits=4))


## 8. Model 4 vs Model 5 Karşılaştırması (CIFAR-10)

In [ ]:
y_true5, y_pred5 = get_preds(model5_cnn, loader_te_c)
acc_cnn5 = accuracy_score(y_true5, y_pred5) * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, yt, yp, title in zip(axes,
        [y_test, y_true5], [y_pred_svm, y_pred5],
        ["Model4: CNN+SVM", "Model5: CNN End-to-End"]):
    cm = confusion_matrix(yt, yp)
    sns.heatmap(cm, annot=True, fmt="d", cmap="YlOrRd", ax=ax,
                xticklabels=CIFAR_CLASSES, yticklabels=CIFAR_CLASSES)
    ax.set(title=title, xlabel="Tahmin", ylabel="Gercek")
    ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.savefig("model45_confusion.png", bbox_inches="tight"); plt.show()

print(f"Model4 (CNN+SVM)         Test: {acc_svm:.2f}%")
print(f"Model5 (CNN End-to-End)  Test: {acc_cnn5:.2f}%")

fig2, ax2 = plt.subplots(figsize=(8,5))
ax2.plot(hist5["train_loss"], "o-", label="Model5 Train Loss")
ax2.set(title="Model5 Egitim Loss (CIFAR-10)", xlabel="Epoch", ylabel="Loss")
ax2.legend(); ax2.grid(True)
plt.savefig("model5_loss.png", bbox_inches="tight"); plt.show()


## 9. Genel Özet Tablosu

In [ ]:
print("="*65)
print(f"{'Model':<30} {'Veri Seti':<12} {'Test Acc':>10}")
print("="*65)
print(f"{'Model1: LeNet-5':<30} {'MNIST':<12} {hist1['test_acc'][-1]:>9.2f}%")
print(f"{'Model2: LeNet5+BN+Drop':<30} {'MNIST':<12} {hist2['test_acc'][-1]:>9.2f}%")
print(f"{'Model3: VGG-11':<30} {'MNIST(sub)':<12} {hist3['test_acc'][-1]:>9.2f}%")
print(f"{'Model4: CNN+SVM':<30} {'CIFAR-10':<12} {acc_svm:>9.2f}%")
print(f"{'Model5: CNN End-to-End':<30} {'CIFAR-10':<12} {acc_cnn5:>9.2f}%")
print("="*65)
